# Project 12: Credit Risk Monitoring (LendingClub)
**Rashi Chaudhary | 2026B15310115**

Run the cells **top to bottom** (Runtime > Run all, or Shift+Enter cell by cell). Each section defines its function and then runs it and shows the result.
Outputs (CSVs, Excel workbook, memo) are saved to `/content/project12_outputs` and zipped for download at the end.

## 0. Setup and settings

In [1]:
import math
import os
import urllib.request
from statistics import NormalDist

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 70)

BASE = os.getcwd()  # /content in Colab

DATA_URL = "https://raw.githubusercontent.com/anushkaparadkar/lending-club-case-study/master/loan.csv"
LOCAL_CSV = os.path.join(BASE, "loan.csv")
OUTPUT_DIR = os.path.join(BASE, "project12_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RNG = np.random.default_rng(42)  # fixed seed so results repeat (re-running a cell advances the generator)
N_BOOT = 1000                    # bootstrap resamples
N_SIM = 20000                    # Monte Carlo loss simulations
TOL = 1.0                        # dollars of tolerance when reconciling balances
MIN_N_FLAG = 100                 # resolved-loan count below which a rate is flagged unreliable
EXPECTED_STATUSES = ["Fully Paid", "Charged Off", "Current"]
GRADE_ORDER = list("ABCDEFG")

COLS = [
    "id", "loan_amnt", "funded_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "home_ownership", "annual_inc", "verification_status", "issue_d", "loan_status", "purpose",
    "addr_state", "dti", "delinq_2yrs", "revol_util", "out_prncp", "total_pymnt",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries",
    "collection_recovery_fee", "last_pymnt_d", "next_pymnt_d",
]
print("Setup done. Output folder:", OUTPUT_DIR)

Setup done. Output folder: /content/project12_outputs


## Helper functions

In [2]:
def wilson(k: int, n: int, z: float = 1.96) -> tuple:
    """Wilson score interval for a proportion."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h


def weighted_quantile(values, weights, q) -> np.ndarray:
    """Quantiles of `values` where each value counts `weights` times."""
    v = np.asarray(values, float)
    w = np.asarray(weights, float)
    ok = np.isfinite(v) & np.isfinite(w)
    v, w = v[ok], w[ok]
    order = np.argsort(v)
    v, w = v[order], w[order]
    cum = (np.cumsum(w) - 0.5 * w) / w.sum()
    return np.interp(np.asarray(q, float), cum, v)


def hhi(shares) -> float:
    """Herfindahl index on a 0-10,000 scale."""
    s = np.asarray(shares, float)
    return float(np.sum((s / s.sum() * 100) ** 2))


def gini(x) -> float:
    x = np.sort(np.asarray(x, float))
    n = len(x)
    return float(2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum()) - (n + 1) / n)


def conc_flag(h: float) -> str:
    # rule-of-thumb cut-offs borrowed from market-concentration practice; a judgment call, not a regulation
    return "high" if h >= 2500 else "moderate" if h >= 1500 else "low"

print("Helpers defined.")

Helpers defined.


## Data import

In [3]:
def load_data() -> pd.DataFrame:
    if not os.path.exists(LOCAL_CSV):
        print("Downloading LendingClub loan data (about 35 MB)...")
        with urllib.request.urlopen(DATA_URL, timeout=120) as resp, open(LOCAL_CSV, "wb") as f:
            f.write(resp.read())
    else:
        print(f"Using local copy: {LOCAL_CSV}")
    return pd.read_csv(LOCAL_CSV, usecols=COLS, low_memory=False)


raw = load_data()
print("Raw shape:", raw.shape)
display(raw.head())

Raw shape: (39717, 27)


,id,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,revol_util,out_prncp,total_pymnt,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,next_pymnt_d
0,1077501,5000,5000,36 months,10.65%,162.87,B,B2,RENT,24000.0,Verified,Dec-11,Fully Paid,credit_card,AZ,27.65,0,83.70%,0.00,5863.155187,5000.00,863.16,0.00,0.00,0.00,Jan-15,NaN
1,1077430,2500,2500,60 months,15.27%,59.83,C,C4,RENT,30000.0,Source Verified,Dec-11,Charged Off,car,GA,1.00,0,9.40%,0.00,1008.710000,456.46,435.17,0.00,117.08,1.11,Apr-13,NaN
2,1077175,2400,2400,36 months,15.96%,84.33,C,C5,RENT,12252.0,Not Verified,Dec-11,Fully Paid,small_business,IL,8.72,0,98.50%,0.00,3005.666844,2400.00,605.67,0.00,0.00,0.00,Jun-14,NaN
3,1076863,10000,10000,36 months,13.49%,339.31,C,C1,RENT,49200.0,Source Verified,Dec-11,Fully Paid,other,CA,20.00,0,21%,0.00,12231.890000,10000.00,2214.92,16.97,0.00,0.00,Jan-15,NaN
4,1075358,3000,3000,60 months,12.69%,67.79,B,B5,RENT,80000.0,Source Verified,Dec-11,Current,other,OR,17.94,0,53.90%,524.06,3513.330000,2475.94,1037.39,0.00,0.00,0.00,May-16,Jun-16


## Step 1: Validate and clean

In [4]:
def step1_validate(raw: pd.DataFrame):
    """Type the columns, run validation rules, drop records that fail the hard rules."""
    df = raw.copy()
    df["loan_status"] = df["loan_status"].astype("string").str.strip()
    df["term_months"] = pd.to_numeric(df["term"].astype("string").str.extract(r"(\d+)")[0], errors="coerce")
    df["int_rate_pct"] = pd.to_numeric(df["int_rate"].astype("string").str.rstrip("%"), errors="coerce")
    df["revol_util_pct"] = pd.to_numeric(df["revol_util"].astype("string").str.rstrip("%"), errors="coerce")
    for src, dst in [("issue_d", "issue_date"), ("last_pymnt_d", "last_pymnt_date"), ("next_pymnt_d", "next_pymnt_date")]:
        df[dst] = pd.to_datetime(df[src], format="%b-%y", errors="coerce")
    money = ["loan_amnt", "funded_amnt", "annual_inc", "out_prncp", "total_pymnt", "total_rec_prncp",
             "total_rec_int", "total_rec_late_fee", "recoveries", "collection_recovery_fee", "installment"]
    for c in money:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    snapshot = df["last_pymnt_date"].max()  # latest payment month in the file = data cut-off

    # (rule, mask of failing rows, hard rule?) - hard failures are removed
    rules = [
        ("Loan ID missing", df["id"].isna(), True),
        ("Loan ID duplicated", df["id"].duplicated(), True),
        ("Loan or funded amount <= 0", (df["loan_amnt"] <= 0) | (df["funded_amnt"] <= 0) | df["funded_amnt"].isna(), True),
        ("Negative balance / payment fields", (df["out_prncp"] < 0) | (df["total_rec_prncp"] < 0) | (df["total_rec_int"] < 0) | (df["recoveries"] < 0), True),
        ("Issue date missing or unparseable", df["issue_date"].isna(), True),
        ("Term not 36 or 60 months", ~df["term_months"].isin([36, 60]), True),
        ("Loan status not in expected set", ~df["loan_status"].isin(EXPECTED_STATUSES), True),
        ("Funded amount > requested amount", df["funded_amnt"] > df["loan_amnt"], False),
        ("Interest rate outside 0-40%", ~df["int_rate_pct"].between(0, 40), False),
        ("Annual income missing or <= 0", df["annual_inc"].isna() | (df["annual_inc"] <= 0), False),
        ("DTI outside 0-100", ~df["dti"].between(0, 100), False),
        ("Revolving utilisation > 100% (can be genuine)", df["revol_util_pct"] > 100, False),
        ("Last payment before issue date", df["last_pymnt_date"] < df["issue_date"], False),
        ("No payment date but not charged off", df["last_pymnt_date"].isna() & (df["loan_status"] != "Charged Off"), False),
        ("Closed loan still showing principal outstanding", (df["loan_status"] != "Current") & (df["out_prncp"] > 0), False),
        ("Current loan with zero principal outstanding", (df["loan_status"] == "Current") & (df["out_prncp"] <= 0), False),
        ("Current loan with no next payment date", (df["loan_status"] == "Current") & df["next_pymnt_date"].isna(), False),
    ]
    audit = pd.DataFrame(
        [{"check": n, "rows_failing": int(m.sum()), "pct_of_loans": round(100 * m.mean(), 3),
          "treatment": "removed" if hard else "flagged only"} for n, m, hard in rules]
    )
    hard_fail = np.zeros(len(df), bool)
    for _, m, hard in rules:
        if hard:
            hard_fail |= m.fillna(False).to_numpy(dtype=bool)
    df = df.loc[~hard_fail].copy()

    status_counts = (
        df.groupby("loan_status").agg(loans=("id", "size"), funded_usd=("funded_amnt", "sum")).reset_index()
    )
    status_counts["pct_of_loans"] = (100 * status_counts["loans"] / status_counts["loans"].sum()).round(2)
    return df, audit, status_counts, snapshot


df, audit, status_counts, snapshot = step1_validate(raw)
n_removed = len(raw) - len(df)
print(f"Data cut-off (latest payment month): {snapshot:%b %Y}")
print(f"Loans removed by hard rules: {n_removed:,}  |  Loans kept: {len(df):,}")
display(audit)
display(status_counts)

Data cut-off (latest payment month): May 2016
Loans removed by hard rules: 0  |  Loans kept: 39,717


,check,rows_failing,pct_of_loans,treatment
0,Loan ID missing,0,0.0,removed
1,Loan ID duplicated,0,0.0,removed
2,Loan or funded amount <= 0,0,0.0,removed
3,Negative balance / payment fields,0,0.0,removed
4,Issue date missing or unparseable,0,0.0,removed
5,Term not 36 or 60 months,0,0.0,removed
6,Loan status not in expected set,0,0.0,removed
7,Funded amount > requested amount,0,0.0,flagged only
8,Interest rate outside 0-40%,0,0.0,flagged only
9,Annual income missing or <= 0,0,0.0,flagged only


,loan_status,loans,funded_usd,pct_of_loans
0,Charged Off,5627,66136375,14.17
1,Current,1140,18793700,2.87
2,Fully Paid,32950,349880250,82.96


## Step 2: Reconcile balances and compare delinquency definitions

In [5]:
def step2_reconcile(df: pd.DataFrame):
    """Tie principal and cash fields together and compare delinquency definitions."""
    df = df.copy()
    df["principal_gap"] = df["funded_amnt"] - df["total_rec_prncp"] - df["out_prncp"]
    parts = df["total_rec_prncp"] + df["total_rec_int"] + df["total_rec_late_fee"] + df["recoveries"]
    df["cash_gap"] = df["total_pymnt"] - parts

    expected = {
        "Fully Paid": "funded = repaid (gap ~ 0)",
        "Current": "funded = repaid + outstanding (gap ~ 0)",
        "Charged Off": "gap = principal written off (>= 0)",
    }
    rows = []
    for status, g in df.groupby("loan_status"):
        rows.append(
            {
                "loan_status": status,
                "loans": len(g),
                "funded_usd": round(g["funded_amnt"].sum()),
                "principal_repaid_usd": round(g["total_rec_prncp"].sum()),
                "outstanding_usd": round(g["out_prncp"].sum()),
                "gap_usd": round(g["principal_gap"].sum()),
                "loans_gap_over_$1": int((g["principal_gap"].abs() > TOL).sum()),
                "loans_negative_gap": int((g["principal_gap"] < -TOL).sum()),
                "cash_identity_breaks": int((g["cash_gap"].abs() > TOL).sum()),
                "expectation": expected[status],
            }
        )
    status_recon = pd.DataFrame(rows)

    co = df["loan_status"] == "Charged Off"
    roll = pd.DataFrame(
        [
            ("Principal originated (funded)", df["funded_amnt"].sum()),
            ("Principal repaid", df["total_rec_prncp"].sum()),
            ("Principal still outstanding", df["out_prncp"].sum()),
            ("Principal written off (charged-off loans)", df.loc[co, "principal_gap"].sum()),
            ("Unexplained gap on non-charged-off loans", df.loc[~co, "principal_gap"].sum()),
            ("Interest received", df["total_rec_int"].sum()),
            ("Late fees received", df["total_rec_late_fee"].sum()),
            ("Recoveries received", df["recoveries"].sum()),
            ("Collection fees on recoveries", df["collection_recovery_fee"].sum()),
        ],
        columns=["item", "usd"],
    )
    roll["usd"] = roll["usd"].round()

    # three things people call "delinquent" - they are not the same
    d_co = co
    d_late = df["total_rec_late_fee"] > 0
    d_prior = df["delinq_2yrs"].fillna(0) > 0
    definitions = pd.DataFrame(
        [
            ("Charged off (loan_status)", int(d_co.sum()), "Terminal default; the cleanest definition here. Not the same as 90+ days past due."),
            ("Ever paid a late fee (total_rec_late_fee > 0)", int(d_late.sum()), "Loan-level lateness during its life; only a partial early-warning signal."),
            ("Borrower had a delinquency in the 2 years BEFORE origination (delinq_2yrs > 0)", int(d_prior.sum()), "A borrower attribute at origination, not performance of this loan."),
        ],
        columns=["definition", "loans_flagged", "comment"],
    )
    definitions["pct_of_loans"] = (100 * definitions["loans_flagged"] / len(df)).round(2)

    overlap = pd.DataFrame(
        {"charged_off": d_co, "late_fee": d_late, "prior_delinq": d_prior}
    ).value_counts().rename("loans").reset_index().sort_values("loans", ascending=False)
    overlap["pct_of_loans"] = (100 * overlap["loans"] / len(df)).round(2)

    caveats = pd.DataFrame(
        [
            ("Days past due", "Not in the file. 30/60/90+ DPD buckets cannot be built; a payment-behaviour bucket is used instead."),
            ("Delinquent status", "Only Current, Fully Paid, Charged Off appear. No 'Late (16-30)' or 'In Grace Period' rows in this extract."),
            ("Snapshot", "Single cut-off (latest payment month). No monthly balance history."),
        ],
        columns=["topic", "note"],
    )
    return df, status_recon, roll, definitions, overlap, caveats


df, status_recon, rollfwd, definitions, overlap, recon_caveats = step2_reconcile(df)
display(status_recon)
display(rollfwd)
display(definitions[["definition", "loans_flagged", "pct_of_loans"]])
display(recon_caveats)

,loan_status,loans,funded_usd,principal_repaid_usd,outstanding_usd,gap_usd,loans_gap_over_$1,loans_negative_gap,cash_identity_breaks,expectation
0,Charged Off,5627,66136375,22323310,0,43813065,5627,0,0,gap = principal written off (>= 0)
1,Current,1140,18793700,16759082,2034618,0,0,0,0,funded = repaid + outstanding (gap ~ 0)
2,Fully Paid,32950,349880250,349880042,0,208,11,0,0,funded = repaid (gap ~ 0)


,item,usd
0,Principal originated (funded),434810325.0
1,Principal repaid,388962435.0
2,Principal still outstanding,2034618.0
3,Principal written off (charged-off loans),43813065.0
4,Unexplained gap on non-charged-off loans,207.0
5,Interest received,89905910.0
6,Late fees received,54135.0
7,Recoveries received,3781917.0
8,Collection fees on recoveries,492734.0


,definition,loans_flagged,pct_of_loans
0,Charged off (loan_status),5627,14.17
1,Ever paid a late fee (total_rec_late_fee > 0),2046,5.15
2,Borrower had a delinquency in the 2 years BEFORE origination (deli...,4312,10.86


,topic,note
0,Days past due,Not in the file. 30/60/90+ DPD buckets cannot be built; a payment-...
1,Delinquent status,"Only Current, Fully Paid, Charged Off appear. No 'Late (16-30)' or..."
2,Snapshot,Single cut-off (latest payment month). No monthly balance history.


## Step 3: Feature engineering

In [6]:
def step3_features(df: pd.DataFrame, snapshot: pd.Timestamp):
    df = df.copy()
    df["vintage_year"] = df["issue_date"].dt.year.astype(int)
    df["vintage_quarter"] = df["issue_date"].dt.to_period("Q").astype(str)

    end = df["last_pymnt_date"].fillna(df["issue_date"])
    end = end.where(df["loan_status"] != "Current", snapshot)  # live loans age to the snapshot
    df["months_on_book"] = (end.dt.year - df["issue_date"].dt.year) * 12 + (end.dt.month - df["issue_date"].dt.month)
    df["months_since_issue"] = (snapshot.year - df["issue_date"].dt.year) * 12 + (snapshot.month - df["issue_date"].dt.month)

    df["is_resolved"] = df["loan_status"] != "Current"
    df["is_default"] = (df["loan_status"] == "Charged Off").astype(int)

    # loss measures (charged-off loans only)
    co = df["is_default"] == 1
    df["gross_loss"] = np.where(co, df["principal_gap"].clip(lower=0), 0.0)
    net_rec = (df["recoveries"] - df["collection_recovery_fee"]).clip(lower=0)
    df["net_loss"] = np.where(co, (df["gross_loss"] - net_rec).clip(lower=0), 0.0)

    df["dti_band"] = pd.cut(df["dti"], [-0.01, 10, 20, 25, 100], labels=["0-10", "10-20", "20-25", "25+"])
    df["income_band"] = pd.cut(df["annual_inc"], [0, 40000, 60000, 80000, 120000, np.inf],
                               labels=["<40k", "40-60k", "60-80k", "80-120k", "120k+"])
    df["size_band"] = pd.cut(df["funded_amnt"], [0, 5000, 10000, 20000, 50000],
                             labels=["<=5k", "5-10k", "10-20k", "20k+"])
    prior = df["delinq_2yrs"].fillna(0)
    df["prior_delinq_bucket"] = np.select([prior == 0, prior == 1], ["0", "1"], default="2+")

    late = df["total_rec_late_fee"] > 0
    df["behaviour_bucket"] = np.select(
        [co, (df["loan_status"] == "Current") & late, df["loan_status"] == "Current", late],
        ["5 Charged off", "3 Current, paid late before", "1 Current, clean", "4 Paid off, paid late before"],
        default="2 Paid off, clean",
    )

    features = pd.DataFrame(
        [
            ("vintage_year / vintage_quarter", "Calendar year / quarter of issue date"),
            ("months_on_book", "Months from issue to last payment (to the snapshot for live loans)"),
            ("months_since_issue", "Months from issue to the data snapshot (seasoning)"),
            ("is_resolved", "Loan is Fully Paid or Charged Off (not still Current)"),
            ("is_default", "Loan status is Charged Off"),
            ("gross_loss", "Principal written off on charged-off loans"),
            ("net_loss", "Gross loss minus recoveries net of collection fees (approximation)"),
            ("behaviour_bucket", "Proxy delinquency bucket: clean / paid late before / charged off"),
            ("prior_delinq_bucket", "Borrower's delinquencies in the 2 years before origination: 0, 1, 2+"),
            ("dti_band, income_band, size_band", "Banded risk and exposure attributes"),
        ],
        columns=["feature", "definition"],
    )
    return df, features


df, features = step3_features(df, snapshot)
display(features)
display(df["behaviour_bucket"].value_counts().sort_index().to_frame("loans"))

,feature,definition
0,vintage_year / vintage_quarter,Calendar year / quarter of issue date
1,months_on_book,Months from issue to last payment (to the snapshot for live loans)
2,months_since_issue,Months from issue to the data snapshot (seasoning)
3,is_resolved,Loan is Fully Paid or Charged Off (not still Current)
4,is_default,Loan status is Charged Off
5,gross_loss,Principal written off on charged-off loans
6,net_loss,Gross loss minus recoveries net of collection fees (approximation)
7,behaviour_bucket,Proxy delinquency bucket: clean / paid late before / charged off
8,prior_delinq_bucket,"Borrower's delinquencies in the 2 years before origination: 0, 1, 2+"
9,"dti_band, income_band, size_band",Banded risk and exposure attributes


,loans
behaviour_bucket,
"1 Current, clean",1089
"2 Paid off, clean",31818
"3 Current, paid late before",51
"4 Paid off, paid late before",1132
5 Charged off,5627


## Step 4: Portfolio loss statistics (NumPy) with bootstrap CIs

In [7]:
def _loss_summary(sub: pd.DataFrame, with_ci: bool) -> list:
    res = sub[sub["is_resolved"]]
    y = res["is_default"].to_numpy(float)
    w = res["funded_amnt"].to_numpy(float)
    funded_co = res.loc[res["is_default"] == 1, "funded_amnt"].sum()

    out = [
        ("Loans", len(sub), None, None),
        ("Funded (USD)", sub["funded_amnt"].sum(), None, None),
        ("Outstanding principal (USD)", sub["out_prncp"].sum(), None, None),
        ("Share of loans resolved (%)", 100 * len(res) / len(sub), None, None),
        ("Avg interest rate, funded-weighted (%)", np.average(sub["int_rate_pct"], weights=sub["funded_amnt"]), None, None),
    ]
    rate_c, rate_w = 100 * y.mean(), 100 * (y * w).sum() / w.sum()
    ci_c = ci_w = (None, None)
    if with_ci:
        n = len(y)
        boot_c, boot_w = np.empty(N_BOOT), np.empty(N_BOOT)
        for i in range(N_BOOT):
            idx = RNG.integers(0, n, n)
            boot_c[i] = y[idx].mean()
            boot_w[i] = (y[idx] * w[idx]).sum() / w[idx].sum()
        ci_c = tuple(np.round(100 * np.percentile(boot_c, [2.5, 97.5]), 2))
        ci_w = tuple(np.round(100 * np.percentile(boot_w, [2.5, 97.5]), 2))
    out += [
        ("Charge-off rate, count-weighted (% of resolved)", rate_c, *ci_c),
        ("Charge-off rate, exposure-weighted (% of resolved funded)", rate_w, *ci_w),
        ("Gross loss rate (% of resolved funded)", 100 * res["gross_loss"].sum() / w.sum(), None, None),
        ("Net loss rate (% of resolved funded)", 100 * res["net_loss"].sum() / w.sum(), None, None),
        ("Gross LGD (% of charged-off funded)", 100 * res["gross_loss"].sum() / funded_co if funded_co else np.nan, None, None),
        ("Net LGD (% of charged-off funded)", 100 * res["net_loss"].sum() / funded_co if funded_co else np.nan, None, None),
        ("Interest received / net loss (x)", res["total_rec_int"].sum() / res["net_loss"].sum() if res["net_loss"].sum() else np.nan, None, None),
    ]
    return [{"metric": m, "value": round(float(v), 2), "ci_low": lo, "ci_high": hi} for m, v, lo, hi in out]


def step4_numpy(df: pd.DataFrame):
    frames = []
    for label, sub in [("All loans", df), ("36 months", df[df["term_months"] == 36]), ("60 months", df[df["term_months"] == 60])]:
        part = pd.DataFrame(_loss_summary(sub, with_ci=True))
        part.insert(0, "segment", label)
        frames.append(part)
    stats = pd.concat(frames, ignore_index=True)

    qs = [5, 25, 50, 75, 95, 99]
    rows = []
    for col, label in [("int_rate_pct", "Interest rate (%)"), ("dti", "DTI"), ("annual_inc", "Annual income (USD)"),
                       ("funded_amnt", "Funded amount (USD)"), ("revol_util_pct", "Revolving utilisation (%)")]:
        v = df[col].to_numpy(float)
        ok = np.isfinite(v)
        plain = np.percentile(v[ok], qs)
        wtd = weighted_quantile(v, df["funded_amnt"], np.array(qs) / 100)
        for q, a, b in zip(qs, plain, wtd):
            rows.append({"variable": label, "percentile": q, "loan_weighted": round(float(a), 2), "funded_weighted": round(float(b), 2)})
    quantiles = pd.DataFrame(rows)
    return stats, quantiles


stats, quantiles = step4_numpy(df)
display(stats)
display(quantiles)

,segment,metric,value,ci_low,ci_high
0,All loans,Loans,3.971700e+04,NaN,NaN
1,All loans,Funded (USD),4.348103e+08,NaN,NaN
2,All loans,Outstanding principal (USD),2.034618e+06,NaN,NaN
3,All loans,Share of loans resolved (%),9.713000e+01,NaN,NaN
4,All loans,"Avg interest rate, funded-weighted (%)",1.279000e+01,NaN,NaN
5,All loans,"Charge-off rate, count-weighted (% of resolved)",1.459000e+01,14.24,14.92
6,All loans,"Charge-off rate, exposure-weighted (% of resolved funded)",1.590000e+01,15.45,16.31
7,All loans,Gross loss rate (% of resolved funded),1.053000e+01,NaN,NaN
8,All loans,Net loss rate (% of resolved funded),9.740000e+00,NaN,NaN
9,All loans,Gross LGD (% of charged-off funded),6.625000e+01,NaN,NaN


,variable,percentile,loan_weighted,funded_weighted
0,Interest rate (%),5,6.17,6.62
1,Interest rate (%),25,9.25,10.25
2,Interest rate (%),50,11.86,12.67
3,Interest rate (%),75,14.59,15.57
4,Interest rate (%),95,18.54,19.69
5,Interest rate (%),99,20.99,21.74
6,DTI,5,2.13,2.50
7,DTI,25,8.17,8.68
8,DTI,50,13.40,13.72
9,DTI,75,18.60,18.70


## Step 5: Performance by segment and vintage

In [8]:
def _perf_rows(df: pd.DataFrame, col: str) -> list:
    rows = []
    for g, sub in df.groupby(col, observed=True):
        res = sub[sub["is_resolved"]]
        n_res, k = len(res), int(res["is_default"].sum())
        lo, hi = wilson(k, n_res)
        res_funded = res["funded_amnt"].sum()
        avg_rate = np.average(sub["int_rate_pct"], weights=sub["funded_amnt"])
        net_loss_pct = 100 * res["net_loss"].sum() / res_funded if res_funded else np.nan
        term_yrs = np.average(res["term_months"], weights=res["funded_amnt"]) / 12 if n_res else np.nan
        rows.append(
            {
                "dimension": col,
                "group": str(g),
                "loans": len(sub),
                "funded_usd": round(sub["funded_amnt"].sum()),
                "pct_unresolved": round(100 * (1 - n_res / len(sub)), 1),
                "charge_off_rate_pct": round(100 * k / n_res, 1) if n_res else np.nan,
                "ci_low_pct": round(100 * lo, 1) if n_res else np.nan,
                "ci_high_pct": round(100 * hi, 1) if n_res else np.nan,
                "exposure_wtd_co_rate_pct": round(100 * (res["is_default"] * res["funded_amnt"]).sum() / res_funded, 1) if res_funded else np.nan,
                "net_loss_pct_of_funded": round(net_loss_pct, 1),
                "avg_interest_rate_pct": round(avg_rate, 2),
                # crude margin: interest rate minus loss spread over the life of the loan
                "margin_after_losses_pp": round(avg_rate - net_loss_pct / term_yrs, 1) if n_res else np.nan,
                "small_n": n_res < MIN_N_FLAG,
            }
        )
    return rows


def step5_performance(df: pd.DataFrame):
    dims = ["vintage_year", "vintage_quarter", "term_months", "purpose", "grade", "dti_band",
            "income_band", "verification_status", "home_ownership", "size_band"]
    perf = pd.DataFrame([r for d in dims for r in _perf_rows(df, d)])

    res = df[df["is_resolved"]]
    # like-for-like view: only loans of the same term are compared across vintages
    vt_rate = res.pivot_table(index="vintage_year", columns="term_months", values="is_default", aggfunc="mean").mul(100).round(1)
    vt_rate.columns = [f"co_rate_pct_{int(c)}m" for c in vt_rate.columns]
    unres = df.assign(u=~df["is_resolved"]).pivot_table(index="vintage_year", columns="term_months", values="u", aggfunc="mean").mul(100).round(1)
    unres.columns = [f"pct_unresolved_{int(c)}m" for c in unres.columns]
    vintage_term = vt_rate.join(unres).reset_index()

    # 36-month loans are fully resolved, so vintage x grade is a clean comparison
    r36 = res[res["term_months"] == 36]
    vintage_grade_36 = r36.pivot_table(index="vintage_year", columns="grade", values="is_default", aggfunc="mean").mul(100).round(1).reset_index()

    # did the grade mix shift between vintages? (mix changes can mimic performance changes)
    mix = df.pivot_table(index="vintage_year", columns="grade", values="funded_amnt", aggfunc="sum").fillna(0)
    grade_mix = mix.div(mix.sum(axis=1), axis=0).mul(100).round(1).reset_index()
    return perf, vintage_term, vintage_grade_36, grade_mix


perf, vintage_term, vintage_grade_36, grade_mix = step5_performance(df)
show = perf[perf["dimension"].isin(["vintage_year", "term_months", "grade", "purpose"])]
display(show.drop(columns=["dimension", "ci_low_pct", "ci_high_pct"]))
print("Vintage x term (like-for-like):")
display(vintage_term)
print("36-month charge-off % by vintage x grade:")
display(vintage_grade_36)
print("Grade mix by vintage (% of funded):")
display(grade_mix)

,group,loans,funded_usd,pct_unresolved,charge_off_rate_pct,exposure_wtd_co_rate_pct,net_loss_pct_of_funded,avg_interest_rate_pct,margin_after_losses_pp,small_n
0,2007,251,2152175,0.0,17.9,21.5,11.7,10.73,6.8,False
1,2008,1562,13457075,0.0,15.8,18.7,11.1,11.53,7.8,False
2,2009,4716,46324425,0.0,12.6,13.4,8.1,12.59,9.9,False
3,2010,11532,116635400,0.0,12.9,13.1,7.7,12.20,10.1,False
4,2011,21656,256241250,5.3,15.9,17.5,10.9,13.17,10.4,False
24,36,29096,275513125,0.0,11.1,10.8,6.1,11.33,9.3,False
25,60,10621,159297200,10.7,25.3,25.8,16.9,15.31,11.9,False
26,car,1549,10400450,3.2,10.7,11.5,6.1,10.82,9.2,False
27,credit_card,5130,58749550,2.0,10.8,12.0,7.0,12.45,10.5,False
28,debt_consolidation,18641,230937800,3.1,15.3,16.5,10.1,13.19,10.5,False


Vintage x term (like-for-like):


,vintage_year,co_rate_pct_36m,co_rate_pct_60m,pct_unresolved_36m,pct_unresolved_60m
0,2007,17.9,NaN,0.0,<NA>
1,2008,15.8,NaN,0.0,<NA>
2,2009,12.6,NaN,0.0,<NA>
3,2010,9.9,21.0,0.0,0.0
4,2011,10.6,27.4,0.0,15.1


36-month charge-off % by vintage x grade:


grade,vintage_year,A,B,C,D,E,F,G
0,2007,1.8,13.1,18.7,40.5,35.7,28.6,NaN
1,2008,5.8,14.6,17.8,24.3,16.0,47.6,50.0
2,2009,6.7,11.4,14.8,17.2,21.6,17.2,34.8
3,2010,4.4,9.3,12.7,16.1,19.2,30.8,36.8
4,2011,6.4,10.6,15.5,18.1,20.6,24.1,40.0


Grade mix by vintage (% of funded):


grade,vintage_year,A,B,C,D,E,F,G
0,2007,14.5,26.0,30.7,16.5,8.8,3.5,0.0
1,2008,13.4,32.8,27.0,16.9,7.1,2.3,0.6
2,2009,18.4,32.2,25.1,15.6,6.3,1.7,0.7
3,2010,19.4,31.9,20.8,15.2,8.3,2.9,1.4
4,2011,20.1,28.7,18.4,14.2,11.6,5.4,1.6


## Step 6: Concentration, loss concentration, roll proxies, charge-off timing

In [9]:
def step6_concentration(df: pd.DataFrame):
    live = df[df["loan_status"] == "Current"]
    dims = ["addr_state", "purpose", "grade", "vintage_year", "term_months", "home_ownership", "verification_status"]

    summary_rows, group_rows = [], []
    for basis, frame, wcol in [("Originated (funded USD)", df, "funded_amnt"), ("Live book (outstanding USD)", live, "out_prncp")]:
        for d in dims:
            s = frame.groupby(d, observed=True)[wcol].sum().sort_values(ascending=False)
            share = s / s.sum()
            h = hhi(s.to_numpy())
            summary_rows.append(
                {
                    "basis": basis, "dimension": d, "groups": len(s),
                    "top1": str(s.index[0]), "top1_share_pct": round(100 * share.iloc[0], 1),
                    "top3_share_pct": round(100 * share.iloc[:3].sum(), 1),
                    "top5_share_pct": round(100 * share.iloc[:5].sum(), 1),
                    "hhi": round(h), "effective_groups": round(10000 / h, 1),
                    # with only 1-2 groups an HHI flag says nothing useful
                    "flag": conc_flag(h) if len(s) > 2 else "n/a (<=2 groups)",
                }
            )
            for g, v in s.items():
                group_rows.append({"basis": basis, "dimension": d, "group": str(g), "exposure_usd": round(v), "share_pct": round(100 * v / s.sum(), 2)})
    conc_summary = pd.DataFrame(summary_rows)
    conc_groups = pd.DataFrame(group_rows)

    # single-name concentration
    sz = df["funded_amnt"].to_numpy(float)
    top_n = np.sort(sz)[::-1]
    single = pd.DataFrame(
        [
            ("Loans", len(sz)),
            ("Largest loan (USD)", top_n[0]),
            ("Top 10 loans, share of funded (%)", 100 * top_n[:10].sum() / sz.sum()),
            ("Top 1% of loans, share of funded (%)", 100 * top_n[: max(1, len(sz) // 100)].sum() / sz.sum()),
            ("Gini of loan size", gini(sz)),
        ],
        columns=["measure", "value"],
    )
    single["value"] = single["value"].round(3)

    # loss concentration: does a group hold more of the loss than its share of exposure?
    res = df[df["is_resolved"]]
    loss_rows = []
    for d in ["addr_state", "purpose", "grade", "vintage_year", "term_months"]:
        exp = res.groupby(d, observed=True)["funded_amnt"].sum()
        loss = res.groupby(d, observed=True)["net_loss"].sum()
        for g in exp.index:
            es, ls = exp[g] / exp.sum(), loss[g] / loss.sum()
            loss_rows.append(
                {
                    "dimension": d, "group": str(g), "exposure_share_pct": round(100 * es, 2),
                    "net_loss_share_pct": round(100 * ls, 2), "loss_index": round(ls / es, 2),
                    "material_and_over_indexed": bool(es >= 0.03 and ls / es >= 1.25),
                }
            )
    loss_conc = pd.DataFrame(loss_rows)

    # roll-rate style proxies (single snapshot, not true roll rates)
    rr_rows = []
    for label, col in [("Ever paid a late fee", np.where(df["total_rec_late_fee"] > 0, "Yes", "No")),
                       ("Prior delinquencies (2y before origination)", df["prior_delinq_bucket"].to_numpy())]:
        tmp = df.assign(state=col)
        for g, sub in tmp.groupby("state"):
            r = sub[sub["is_resolved"]]
            k = int(r["is_default"].sum())
            lo, hi = wilson(k, len(r))
            rr_rows.append(
                {
                    "state_variable": label, "state": g, "loans": len(sub),
                    "pct_still_current": round(100 * (1 - len(r) / len(sub)), 1),
                    "charge_off_rate_pct": round(100 * k / len(r), 1),
                    "ci_low_pct": round(100 * lo, 1), "ci_high_pct": round(100 * hi, 1),
                }
            )
    roll_proxy = pd.DataFrame(rr_rows)

    # when do charge-offs happen? (months on book at last payment)
    timing_rows = []
    for term, sub in df[df["is_default"] == 1].groupby("term_months"):
        m = sub["months_on_book"].to_numpy(float)
        p10, p25, p50, p75, p90 = np.percentile(m, [10, 25, 50, 75, 90])
        row = {"term_months": int(term), "charged_off_loans": len(m), "p10": p10, "p25": p25, "median": p50, "p75": p75, "p90": p90}
        for cut in [6, 12, 18, 24, 36]:
            row[f"cum_share_by_{cut}m_pct"] = round(100 * (m <= cut).mean(), 1)
        timing_rows.append(row)
    timing = pd.DataFrame(timing_rows)
    return conc_summary, conc_groups, single, loss_conc, roll_proxy, timing


conc_summary, conc_groups, single, loss_conc, roll_proxy, timing = step6_concentration(df)
display(conc_summary)
display(single)
print("Over-indexed loss segments:")
display(loss_conc[loss_conc["material_and_over_indexed"]])
print("Roll proxies (single snapshot, not true roll rates):")
display(roll_proxy)
print("Charge-off timing (months on book):")
display(timing)

,basis,dimension,groups,top1,top1_share_pct,top3_share_pct,top5_share_pct,hhi,effective_groups,flag
0,Originated (funded USD),addr_state,50,CA,18.0,34.8,46.6,664,15.1,low
1,Originated (funded USD),purpose,14,debt_consolidation,53.1,74.3,87.0,3175,3.1,high
2,Originated (funded USD),grade,7,B,30.0,69.6,94.3,2021,4.9,moderate
3,Originated (funded USD),vintage_year,5,2011,58.9,96.4,100.0,4316,2.3,high
4,Originated (funded USD),term_months,2,36,63.4,100.0,100.0,5357,1.9,n/a (<=2 groups)
5,Originated (funded USD),home_ownership,5,MORTGAGE,50.0,99.8,100.0,4374,2.3,high
6,Originated (funded USD),verification_status,3,Verified,45.0,100.0,100.0,3585,2.8,high
7,Live book (outstanding USD),addr_state,43,CA,12.6,28.5,40.3,523,19.1,low
8,Live book (outstanding USD),purpose,13,debt_consolidation,57.7,77.6,91.8,3652,2.7,high
9,Live book (outstanding USD),grade,7,B,28.5,69.0,95.8,2034,4.9,moderate


,measure,value
0,Loans,39717.000
1,Largest loan (USD),35000.000
2,"Top 10 loans, share of funded (%)",0.080
3,"Top 1% of loans, share of funded (%)",3.196
4,Gini of loan size,0.356


Over-indexed loss segments:


,dimension,group,exposure_share_pct,net_loss_share_pct,loss_index,material_and_over_indexed
9,addr_state,FL,6.89,8.90,1.29,True
61,purpose,small_business,5.42,10.82,2.00,True
67,grade,D,14.59,20.60,1.41,True
68,grade,E,9.67,17.56,1.82,True
69,grade,F,4.10,9.57,2.34,True
77,term_months,60,33.77,58.47,1.73,True


Roll proxies (single snapshot, not true roll rates):


,state_variable,state,loans,pct_still_current,charge_off_rate_pct,ci_low_pct,ci_high_pct
0,Ever paid a late fee,No,37671,2.9,13.0,12.7,13.4
1,Ever paid a late fee,Yes,2046,2.5,43.3,41.1,45.4
2,Prior delinquencies (2y before origination),0,35405,2.9,14.4,14.0,14.7
3,Prior delinquencies (2y before origination),1,3303,2.9,15.8,14.6,17.1
4,Prior delinquencies (2y before origination),2+,1009,2.5,18.7,16.4,21.3


Charge-off timing (months on book):


,term_months,charged_off_loans,p10,p25,median,p75,p90,cum_share_by_6m_pct,cum_share_by_12m_pct,cum_share_by_18m_pct,cum_share_by_24m_pct,cum_share_by_36m_pct
0,36,3227,4.0,9.0,16.0,24.0,30.0,16.8,39.1,58.4,76.5,99.0
1,60,2400,6.0,11.0,20.0,34.0,44.0,12.3,29.7,45.0,59.8,80.1


## Step 7: Stress tests, sensitivity, reverse stress, Monte Carlo

In [10]:
def _segment_params(df: pd.DataFrame) -> pd.DataFrame:
    """PD, LGD and exposure per grade x term, estimated from resolved loans."""
    res = df[df["is_resolved"]]
    seg = res.groupby(["grade", "term_months"]).agg(
        resolved=("is_default", "size"), defaults=("is_default", "sum"),
        funded_co=("funded_amnt", lambda s: 0.0),
    ).reset_index()
    co = res[res["is_default"] == 1].groupby(["grade", "term_months"]).agg(
        funded_co=("funded_amnt", "sum"), gross=("gross_loss", "sum"), net=("net_loss", "sum")
    ).reset_index()
    seg = seg.drop(columns="funded_co").merge(co, on=["grade", "term_months"], how="left").fillna(0)
    overall_net_lgd = res["net_loss"].sum() / res.loc[res["is_default"] == 1, "funded_amnt"].sum()
    overall_gross_lgd = res["gross_loss"].sum() / res.loc[res["is_default"] == 1, "funded_amnt"].sum()
    seg["pd"] = seg["defaults"] / seg["resolved"]
    seg["lgd_net"] = np.where(seg["funded_co"] > 0, seg["net"] / seg["funded_co"].replace(0, np.nan), overall_net_lgd)
    seg["lgd_gross"] = np.where(seg["funded_co"] > 0, seg["gross"] / seg["funded_co"].replace(0, np.nan), overall_gross_lgd)

    book = df.groupby(["grade", "term_months"]).agg(book_loans=("id", "size"), ead=("funded_amnt", "sum")).reset_index()
    seg = seg.merge(book, on=["grade", "term_months"], how="right")
    seg["avg_ead"] = seg["ead"] / seg["book_loans"]
    return seg


def step7_stress(df: pd.DataFrame):
    seg = _segment_params(df)
    key = df["grade"] + "|" + df["term_months"].astype(int).astype(str)
    seg_key = seg["grade"] + "|" + seg["term_months"].astype(int).astype(str)
    pd_map = dict(zip(seg_key, seg["pd"]))
    lgd_map = dict(zip(seg_key, seg["lgd_net"]))
    lgdg_map = dict(zip(seg_key, seg["lgd_gross"]))

    ead = df["funded_amnt"].to_numpy(float)
    pd0 = key.map(pd_map).to_numpy(float)
    lgd0 = key.map(lgd_map).to_numpy(float)
    lgd_gross = key.map(lgdg_map).to_numpy(float)

    # PD of the next-worse grade, same term (for the migration scenario)
    next_grade = {g: GRADE_ORDER[min(i + 1, len(GRADE_ORDER) - 1)] for i, g in enumerate(GRADE_ORDER)}
    nxt_key = df["grade"].map(next_grade) + "|" + df["term_months"].astype(int).astype(str)
    pd_next = nxt_key.map(pd_map).fillna(pd.Series(pd0, index=df.index)).to_numpy(float)
    ab_c = df["grade"].isin(["A", "B", "C"]).to_numpy()

    top_state = df.groupby("addr_state")["funded_amnt"].sum().idxmax()
    in_top_state = (df["addr_state"] == top_state).to_numpy()
    high_dti = (df["dti"] >= 20).to_numpy()

    def el(pd_mult=1.0, lgd_add=0.0, zero_recoveries=False, migrate=0.0, state_mult=1.0, dti_mult=1.0):
        p = pd0.copy()
        if migrate:
            p = np.where(ab_c, (1 - migrate) * p + migrate * pd_next, p)
        p = p * pd_mult * np.where(in_top_state, state_mult, 1.0) * np.where(high_dti, dti_mult, 1.0)
        p = np.clip(p, 0, 1)
        lg = lgd_gross if zero_recoveries else lgd0
        lg = np.clip(lg + lgd_add, 0, 1)
        return float((ead * p * lg).sum())

    base = el()
    total = ead.sum()
    scenarios = [
        ("Base (observed segment PD x LGD)", "No stress", {}),
        ("PD +25%", "All segment default rates x1.25", dict(pd_mult=1.25)),
        ("PD +50%", "All segment default rates x1.5", dict(pd_mult=1.5)),
        ("PD x2 (severe)", "All segment default rates doubled", dict(pd_mult=2.0)),
        ("LGD +10 pts", "Recoveries weaken; net LGD up 10 percentage points", dict(lgd_add=0.10)),
        ("No recoveries", "Recovery income falls to zero; LGD = gross LGD", dict(zero_recoveries=True)),
        ("Grade migration (20% of A-C one grade worse)", "20% of A-C exposure takes next grade's PD", dict(migrate=0.20)),
        (f"Concentration shock ({top_state} PD x1.5)", "Largest state by exposure suffers a regional downturn", dict(state_mult=1.5)),
        ("High-DTI shock (DTI>=20 PD x1.3)", "Stretched borrowers deteriorate", dict(dti_mult=1.3)),
        ("Combined: PD +50% and LGD +10 pts", "Joint deterioration", dict(pd_mult=1.5, lgd_add=0.10)),
        ("Severe combined: PD x2, no recoveries", "Tail-style joint stress", dict(pd_mult=2.0, zero_recoveries=True)),
    ]
    rows = []
    for name, desc, kw in scenarios:
        v = el(**kw)
        rows.append(
            {
                "scenario": name, "description": desc,
                "expected_loss_usd": round(v),
                "loss_rate_pct_of_exposure": round(100 * v / total, 2),
                "change_vs_base_usd": round(v - base),
                "change_vs_base_pp": round(100 * (v - base) / total, 2),
                "pct_change_vs_base": round(100 * (v / base - 1), 1),
            }
        )
    scen = pd.DataFrame(rows)

    # one-at-a-time sensitivity (tornado style)
    sens_defs = [
        ("PD multiplier", "x0.75", "x1.25", dict(pd_mult=0.75), dict(pd_mult=1.25)),
        ("Net LGD", "-10 pts", "+10 pts", dict(lgd_add=-0.10), dict(lgd_add=0.10)),
        ("Recoveries", "as observed", "zero", {}, dict(zero_recoveries=True)),
        ("Grade migration (A-C)", "0%", "40%", {}, dict(migrate=0.40)),
        (f"{top_state} PD", "x1.0", "x2.0", {}, dict(state_mult=2.0)),
        ("High-DTI PD (DTI>=20)", "x1.0", "x1.5", {}, dict(dti_mult=1.5)),
    ]
    srows = []
    for name, lo_lab, hi_lab, lo_kw, hi_kw in sens_defs:
        lo_v, hi_v = el(**lo_kw), el(**hi_kw)
        srows.append(
            {"assumption": name, "low_case": lo_lab, "high_case": hi_lab,
             "loss_rate_low_pct": round(100 * lo_v / total, 2), "loss_rate_base_pct": round(100 * base / total, 2),
             "loss_rate_high_pct": round(100 * hi_v / total, 2), "swing_pp": round(100 * abs(hi_v - lo_v) / total, 2)}
        )
    sens = pd.DataFrame(srows).sort_values("swing_pp", ascending=False).reset_index(drop=True)

    # reverse stress: how far can defaults rise before losses eat all interest earned?
    res = df[df["is_resolved"]]
    interest, net_loss = res["total_rec_int"].sum(), res["net_loss"].sum()
    breakeven = interest / net_loss
    reverse = pd.DataFrame(
        [("Interest received on resolved loans (USD)", round(interest)),
         ("Net principal loss on resolved loans (USD)", round(net_loss)),
         ("Default-rate multiple at which losses = all interest earned (approx.)", round(breakeven, 2))],
        columns=["measure", "value"],
    )

    # Monte Carlo with one shared "economy" factor (Vasicek-style). rho is an ASSUMPTION.
    nd = NormalDist()
    cdf = np.frompyfunc(lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2))), 1, 1)
    p_seg = seg["pd"].fillna(0).clip(1e-6, 1 - 1e-6).to_numpy()
    inv = np.array([nd.inv_cdf(p) for p in p_seg])
    n_seg = seg["book_loans"].to_numpy(int)
    loss_per_default = (seg["avg_ead"] * seg["lgd_net"]).to_numpy(float)
    z = RNG.standard_normal(N_SIM)
    mc_rows = []
    for rho in [0.0, 0.05, 0.10, 0.20]:
        if rho == 0:
            cond = np.repeat(p_seg[:, None], N_SIM, axis=1)
        else:
            cond = cdf((inv[:, None] - math.sqrt(rho) * z[None, :]) / math.sqrt(1 - rho)).astype(float)
        defaults = RNG.binomial(n_seg[:, None], np.clip(cond, 0, 1))
        loss_rate = 100 * (defaults * loss_per_default[:, None]).sum(axis=0) / total
        p95, p99, p999 = np.percentile(loss_rate, [95, 99, 99.9])
        mc_rows.append(
            {"asset_correlation_rho": rho, "mean_loss_pct": round(loss_rate.mean(), 2),
             "p50_pct": round(np.percentile(loss_rate, 50), 2), "p95_pct": round(p95, 2),
             "p99_pct": round(p99, 2), "p99.9_pct": round(p999, 2),
             "expected_shortfall_99_pct": round(loss_rate[loss_rate >= p99].mean(), 2),
             "unexpected_loss_p99_pct": round(p99 - loss_rate.mean(), 2)}
        )
    mc = pd.DataFrame(mc_rows)

    seg_out = seg[["grade", "term_months", "book_loans", "ead", "resolved", "defaults", "pd", "lgd_net", "lgd_gross"]].copy()
    seg_out["ead"] = seg_out["ead"].round()
    for c in ["pd", "lgd_net", "lgd_gross"]:
        seg_out[c] = seg_out[c].round(4)
    return scen, sens, reverse, mc, seg_out, top_state


scen, sens, reverse, mc, seg_out, top_state = step7_stress(df)
print("Scenarios:")
display(scen.drop(columns="description"))
print("Sensitivity:")
display(sens)
print("Reverse stress:")
display(reverse)
print("Monte Carlo (loss as % of exposure):")
display(mc)

Scenarios:


,scenario,expected_loss_usd,loss_rate_pct_of_exposure,change_vs_base_usd,change_vs_base_pp,pct_change_vs_base
0,Base (observed segment PD x LGD),44822321,10.31,0,0.00,0.0
1,PD +25%,56027901,12.89,11205580,2.58,25.0
2,PD +50%,67233481,15.46,22411160,5.15,50.0
3,PD x2 (severe),89644641,20.62,44822321,10.31,100.0
4,LGD +10 pts,52126858,11.99,7304537,1.68,16.3
5,No recoveries,48454370,11.14,3632049,0.84,8.1
6,Grade migration (20% of A-C one grade worse),46338473,10.66,1516152,0.35,3.4
7,Concentration shock (CA PD x1.5),48781069,11.22,3958748,0.91,8.8
8,High-DTI shock (DTI>=20 PD x1.3),47651140,10.96,2828820,0.65,6.3
9,Combined: PD +50% and LGD +10 pts,78190287,17.98,33367966,7.67,74.4


Sensitivity:


,assumption,low_case,high_case,loss_rate_low_pct,loss_rate_base_pct,loss_rate_high_pct,swing_pp
0,PD multiplier,x0.75,x1.25,7.73,10.31,12.89,5.15
1,Net LGD,-10 pts,+10 pts,8.63,10.31,11.99,3.36
2,CA PD,x1.0,x2.0,10.31,10.31,12.13,1.82
3,High-DTI PD (DTI>=20),x1.0,x1.5,10.31,10.31,11.39,1.08
4,Recoveries,as observed,zero,10.31,10.31,11.14,0.84
5,Grade migration (A-C),0%,40%,10.31,10.31,11.01,0.70


Reverse stress:


,measure,value
0,Interest received on resolved loans (USD),81754052.00
1,Net principal loss on resolved loans (USD),40524632.00
2,Default-rate multiple at which losses = all interest earned (approx.),2.02


Monte Carlo (loss as % of exposure):


,asset_correlation_rho,mean_loss_pct,p50_pct,p95_pct,p99_pct,p99.9_pct,expected_shortfall_99_pct,unexpected_loss_p99_pct
0,0.00,10.31,10.31,10.51,10.59,10.68,10.63,0.29
1,0.05,10.29,9.98,16.08,19.13,22.58,20.68,8.84
2,0.10,10.28,9.65,18.80,23.53,28.74,25.94,13.24
3,0.20,10.27,8.94,23.10,30.62,38.27,34.12,20.35


## Step 8: Limitations and risk memo

In [11]:
LIMITATIONS = [
    ("Selection bias: accepted loans only",
     "The file holds loans that were approved and funded. Rejected applications are absent, so default rates describe the approved population, not all applicants. Any model or cut-off built on it is blind to who was screened out."),
    ("Selection bias: one platform, one era",
     "Loans were issued 2007-2011 through a peer-to-peer platform during and after the financial crisis. This is not a bank balance sheet, and the cycle covered is narrow. Do not extrapolate to other lenders, products or economic conditions."),
    ("Right-censoring",
     "Some 60-month loans issued in 2010-2011 were still active at the May 2016 snapshot. Their charge-off rates are understated because not all outcomes are known. Compare vintages like-for-like (36-month loans) or by months on book."),
    ("No payment history",
     "There are no monthly balances or days-past-due fields. True delinquency buckets, cure rates and roll rates cannot be measured. The late-fee and prior-delinquency views here are proxies, and late fees can occur just before default."),
    ("Definition of default",
     "Charged off is a late, accounting-based event (typically well after 90 days past due) and is not a regulatory default definition."),
    ("Endogenous risk grades",
     "The grade and interest rate were set by the platform's own model and are not independent inputs. Differences by grade partly reflect that model and pricing."),
    ("Self-reported and mixed-quality inputs",
     "Income, purpose and employment details are largely self-reported; verification status varies."),
    ("Third-party copy",
     "The file is a public mirror of platform data, not an audited ledger. Reconcile to source before any formal use."),
    ("Stress tests are illustrative",
     "Scenario multipliers and the Monte Carlo correlation are assumptions, not calibrated forecasts. The stress book assumes today's observed segment PD/LGD apply to the whole origination mix."),
]

DECISION_LIMITS = [
    "Not for individual credit decisions: nothing here scores, approves or declines any applicant.",
    "Fair lending (e.g. ECOA / Regulation B, or local equivalents): state and income patterns can correlate with protected characteristics. Aggregate monitoring is fine; using these cuts to price or decline requires a separate fair-lending review.",
    "Adverse action and credit reporting rules (e.g. FCRA-type notice requirements) apply to real decisions and are not addressed here.",
    "Not an IFRS 9 / CECL / Basel provisioning or capital calculation. Segment PD and LGD here are simple observed averages without forward-looking adjustment or discounting.",
    "Model risk governance (e.g. SR 11-7-style validation) would be needed before any of this feeds a production model or limit.",
    "Use for portfolio monitoring, hypothesis generation and teaching. Re-run on the lender's own data before drawing operational conclusions.",
]

limits = pd.DataFrame(LIMITATIONS, columns=["limitation", "detail"])
display(limits)

,limitation,detail
0,Selection bias: accepted loans only,The file holds loans that were approved and funded. Rejected appli...
1,"Selection bias: one platform, one era",Loans were issued 2007-2011 through a peer-to-peer platform during...
2,Right-censoring,Some 60-month loans issued in 2010-2011 were still active at the M...
3,No payment history,There are no monthly balances or days-past-due fields. True delinq...
4,Definition of default,"Charged off is a late, accounting-based event (typically well afte..."
5,Endogenous risk grades,The grade and interest rate were set by the platform's own model a...
6,Self-reported and mixed-quality inputs,"Income, purpose and employment details are largely self-reported; ..."
7,Third-party copy,"The file is a public mirror of platform data, not an audited ledge..."
8,Stress tests are illustrative,Scenario multipliers and the Monte Carlo correlation are assumptio...


In [12]:
def build_memo(df, stats, perf, vintage_term, conc_summary, loss_conc, roll_proxy, scen, sens, reverse, mc,
               definitions, audit, snapshot, top_state, n_removed) -> str:
    s_all = stats[stats["segment"] == "All loans"].set_index("metric")
    s36 = stats[stats["segment"] == "36 months"].set_index("metric")
    s60 = stats[stats["segment"] == "60 months"].set_index("metric")
    co_key = "Charge-off rate, count-weighted (% of resolved)"
    nl_key = "Net loss rate (% of resolved funded)"

    grades = perf[perf["dimension"] == "grade"].set_index("group")
    purposes = perf[perf["dimension"] == "purpose"].sort_values("charge_off_rate_pct", ascending=False)
    purposes = purposes[~purposes["small_n"]]
    weak_txt = ", ".join(f"{r.group} ({r.charge_off_rate_pct}%)" for r in purposes.head(3).itertuples())

    states = conc_summary[(conc_summary["dimension"] == "addr_state") & conc_summary["basis"].str.startswith("Orig")].iloc[0]
    purp_c = conc_summary[(conc_summary["dimension"] == "purpose") & conc_summary["basis"].str.startswith("Orig")].iloc[0]
    live = df[df["loan_status"] == "Current"]
    live_terms = ", ".join(f"{int(t)}-month" for t in sorted(live["term_months"].unique()))
    live_years = ", ".join(str(int(y)) for y in sorted(live["vintage_year"].unique()))
    live_state = live.groupby("addr_state")["out_prncp"].sum().sort_values(ascending=False)

    vt = vintage_term.dropna(subset=["co_rate_pct_36m"])
    worst_v = vt.loc[vt["co_rate_pct_36m"].idxmax()]
    best_v = vt.loc[vt["co_rate_pct_36m"].idxmin()]

    over = loss_conc[loss_conc["material_and_over_indexed"]].sort_values("loss_index", ascending=False).head(6)
    over_txt = "\n".join(f"- {r.dimension} = {r.group}: {r.exposure_share_pct}% of exposure but {r.net_loss_share_pct}% of net loss (index {r.loss_index})" for r in over.itertuples()) or "- None above the threshold."

    late = roll_proxy[roll_proxy["state_variable"] == "Ever paid a late fee"].set_index("state")
    mc10 = mc[mc["asset_correlation_rho"] == 0.10].iloc[0]
    sc = scen.set_index("scenario")
    base_rate = sc.iloc[0]["loss_rate_pct_of_exposure"]
    breakeven = reverse.iloc[2]["value"]
    top_sens = sens.iloc[0]

    scen_lines = "\n".join(f"| {r.scenario} | {r.loss_rate_pct_of_exposure}% | {r.change_vs_base_pp:+.2f} pp |" for r in scen.itertuples())
    lim_lines = "\n".join(f"- **{t}.** {d}" for t, d in LIMITATIONS)
    dec_lines = "\n".join(f"- {t}" for t in DECISION_LIMITS)
    defs = definitions.set_index("definition")["pct_of_loans"]
    soft = audit[(audit["rows_failing"] > 0) & (audit["treatment"] == "flagged only")]
    soft_txt = "; ".join(f"{r.check} ({r.rows_failing})" for r in soft.itertuples()) or "none"

    return f"""# Credit risk memo: loan portfolio monitoring

*Data: LendingClub public loan file, {len(df):,} loans issued {df['issue_date'].min():%b %Y}-{df['issue_date'].max():%b %Y}, data cut-off {snapshot:%b %Y}. Monitoring and teaching use only.*

## 1. Headline
- Originated **${df['funded_amnt'].sum()/1e6:,.1f}m** across {len(df):,} loans; **{s_all.loc['Share of loans resolved (%)','value']:.1f}%** have a final outcome (paid or charged off). ${df['out_prncp'].sum()/1e6:,.1f}m of principal is still outstanding.
- Charge-off rate on resolved loans: **{s_all.loc[co_key,'value']:.1f}%** by count (95% CI {s_all.loc[co_key,'ci_low']:.1f}-{s_all.loc[co_key,'ci_high']:.1f}%). Net principal loss: **{s_all.loc[nl_key,'value']:.1f}%** of resolved funded principal.
- 60-month loans lose more: {s60.loc[co_key,'value']:.1f}% charge-off vs {s36.loc[co_key,'value']:.1f}% for 36-month loans (60-month figure is also understated by unresolved loans).
- Interest received covers net losses about **{s_all.loc['Interest received / net loss (x)','value']:.1f}x** on resolved loans. Default rates would need to rise about **{breakeven:.1f}x** before losses absorb all interest earned (approximate reverse stress).

## 2. Data quality and reconciliation
- Validation: {n_removed:,} loans failed the hard rules and were removed; soft flags (kept, flagged only): {soft_txt}.
- Principal reconciles for Fully Paid and Current loans to within $1 except where listed in `status_reconciliation.csv`; charged-off loans show the written-off principal as the gap.
- Three "delinquency" definitions disagree: {defs.iloc[0]:.1f}% charged off, {defs.iloc[1]:.1f}% ever paid a late fee, {defs.iloc[2]:.1f}% borrowers had a delinquency before origination. Rates in this memo use **charged off**; late fee is an early-warning proxy only.
- No days-past-due field exists, so 30/60/90+ buckets and true roll rates are not available.

## 3. Performance
- By grade, charge-off rises from **{grades.loc['A','charge_off_rate_pct']}%** (A) to **{grades.loc['G','charge_off_rate_pct']}%** (G); average rate rises from {grades.loc['A','avg_interest_rate_pct']}% to {grades.loc['G','avg_interest_rate_pct']}%.
- Among 36-month loans (fully resolved), charge-off by vintage ranges from **{best_v['co_rate_pct_36m']}%** ({int(best_v['vintage_year'])}) to **{worst_v['co_rate_pct_36m']}%** ({int(worst_v['vintage_year'])}). Check the grade-mix table before reading this as a change in underwriting quality.
- Weakest purposes with enough data: {weak_txt}.
- Loans that paid a late fee were charged off {late.loc['Yes','charge_off_rate_pct']}% of the time vs {late.loc['No','charge_off_rate_pct']}% for those that never did, but most charge-offs had no late fee on record, so it is a weak early warning.

## 4. Concentration
- Largest state: **{states['top1']}** at {states['top1_share_pct']}% of originations (HHI {states['hhi']}, {states['flag']}). Largest purpose: **{purp_c['top1']}** at {purp_c['top1_share_pct']}% (HHI {purp_c['hhi']}, {purp_c['flag']}).
- The live (still outstanding) book is small: ${live['out_prncp'].sum()/1e6:,.1f}m across {len(live):,} loans, all {live_terms} loans issued in {live_years}. Its largest state is {live_state.index[0]} ({100*live_state.iloc[0]/live_state.sum():.1f}%).
 Concentration by term and vintage is trivially 100% and is not a useful signal on its own.
- Segments carrying more loss than their share of exposure:
{over_txt}

## 5. Stress and sensitivity (illustrative)
Loss rate is expected net loss as a percentage of originated exposure, using observed grade x term PD and LGD. The base ({base_rate}%) sits above the observed net loss on resolved loans ({s_all.loc[nl_key,'value']:.1f}%) because still-live 60-month loans are assigned a full-life default rate.

| Scenario | Loss rate | vs base |
|---|---|---|
{scen_lines}

- Most sensitive single assumption: **{top_sens['assumption']}** (swing {top_sens['swing_pp']} pp between low and high case).
- Monte Carlo with an assumed asset correlation of 0.10: mean loss {mc10['mean_loss_pct']}%, 99th percentile {mc10['p99_pct']}%, so unexpected loss at 99% is about {mc10['unexpected_loss_p99_pct']} pp. The correlation is an assumption; see the sheet for 0 to 0.20.

## 6. Selection bias and limitations
{lim_lines}

## 7. Regulatory and decision-use limits
{dec_lines}

## 8. Suggested monitoring actions
1. Track charge-off by vintage on a like-for-like basis (same term, same months on book) and report unresolved share next to every rate.
2. Set concentration triggers for single-state, single-purpose and single-vintage exposure (the HHI flags above are a starting point).
3. Collect monthly days-past-due and balance history so true delinquency buckets, cure rates and roll rates can replace today's proxies.
4. Re-run stress scenarios each period and compare the loss-rate swing by assumption; challenge the biggest ones first.
"""


memo = build_memo(df, stats, perf, vintage_term, conc_summary, loss_conc, roll_proxy, scen, sens,
                  reverse, mc, definitions, audit, snapshot, top_state, n_removed)
display(Markdown(memo))

# Credit risk memo: loan portfolio monitoring

*Data: LendingClub public loan file, 39,717 loans issued Jun 2007-Dec 2011, data cut-off May 2016. Monitoring and teaching use only.*

## 1. Headline
- Originated **$434.8m** across 39,717 loans; **97.1%** have a final outcome (paid or charged off). $2.0m of principal is still outstanding.
- Charge-off rate on resolved loans: **14.6%** by count (95% CI 14.2-14.9%). Net principal loss: **9.7%** of resolved funded principal.
- 60-month loans lose more: 25.3% charge-off vs 11.1% for 36-month loans (60-month figure is also understated by unresolved loans).
- Interest received covers net losses about **2.0x** on resolved loans. Default rates would need to rise about **2.0x** before losses absorb all interest earned (approximate reverse stress).

## 2. Data quality and reconciliation
- Validation: 0 loans failed the hard rules and were removed; soft flags (kept, flagged only): none.
- Principal reconciles for Fully Paid and Current loans to within $1 except where listed in `status_reconciliation.csv`; charged-off loans show the written-off principal as the gap.
- Three "delinquency" definitions disagree: 14.2% charged off, 5.2% ever paid a late fee, 10.9% borrowers had a delinquency before origination. Rates in this memo use **charged off**; late fee is an early-warning proxy only.
- No days-past-due field exists, so 30/60/90+ buckets and true roll rates are not available.

## 3. Performance
- By grade, charge-off rises from **6.0%** (A) to **33.8%** (G); average rate rises from 7.39% to 21.51%.
- Among 36-month loans (fully resolved), charge-off by vintage ranges from **9.9%** (2010) to **17.9%** (2007). Check the grade-mix table before reading this as a change in underwriting quality.
- Weakest purposes with enough data: small_business (27.1%), renewable_energy (18.6%), educational (17.2%).
- Loans that paid a late fee were charged off 43.3% of the time vs 13.0% for those that never did, but most charge-offs had no late fee on record, so it is a weak early warning.

## 4. Concentration
- Largest state: **CA** at 18.0% of originations (HHI 664, low). Largest purpose: **debt_consolidation** at 53.1% (HHI 3175, high).
- The live (still outstanding) book is small: $2.0m across 1,140 loans, all 60-month loans issued in 2011. Its largest state is CA (12.6%).
 Concentration by term and vintage is trivially 100% and is not a useful signal on its own.
- Segments carrying more loss than their share of exposure:
- grade = F: 4.1% of exposure but 9.57% of net loss (index 2.34)
- purpose = small_business: 5.42% of exposure but 10.82% of net loss (index 2.0)
- grade = E: 9.67% of exposure but 17.56% of net loss (index 1.82)
- term_months = 60: 33.77% of exposure but 58.47% of net loss (index 1.73)
- grade = D: 14.59% of exposure but 20.6% of net loss (index 1.41)
- addr_state = FL: 6.89% of exposure but 8.9% of net loss (index 1.29)

## 5. Stress and sensitivity (illustrative)
Loss rate is expected net loss as a percentage of originated exposure, using observed grade x term PD and LGD. The base (10.31%) sits above the observed net loss on resolved loans (9.7%) because still-live 60-month loans are assigned a full-life default rate.

| Scenario | Loss rate | vs base |
|---|---|---|
| Base (observed segment PD x LGD) | 10.31% | +0.00 pp |
| PD +25% | 12.89% | +2.58 pp |
| PD +50% | 15.46% | +5.15 pp |
| PD x2 (severe) | 20.62% | +10.31 pp |
| LGD +10 pts | 11.99% | +1.68 pp |
| No recoveries | 11.14% | +0.84 pp |
| Grade migration (20% of A-C one grade worse) | 10.66% | +0.35 pp |
| Concentration shock (CA PD x1.5) | 11.22% | +0.91 pp |
| High-DTI shock (DTI>=20 PD x1.3) | 10.96% | +0.65 pp |
| Combined: PD +50% and LGD +10 pts | 17.98% | +7.67 pp |
| Severe combined: PD x2, no recoveries | 22.29% | +11.98 pp |

- Most sensitive single assumption: **PD multiplier** (swing 5.15 pp between low and high case).
- Monte Carlo with an assumed asset correlation of 0.10: mean loss 10.28%, 99th percentile 23.53%, so unexpected loss at 99% is about 13.24 pp. The correlation is an assumption; see the sheet for 0 to 0.20.

## 6. Selection bias and limitations
- **Selection bias: accepted loans only.** The file holds loans that were approved and funded. Rejected applications are absent, so default rates describe the approved population, not all applicants. Any model or cut-off built on it is blind to who was screened out.
- **Selection bias: one platform, one era.** Loans were issued 2007-2011 through a peer-to-peer platform during and after the financial crisis. This is not a bank balance sheet, and the cycle covered is narrow. Do not extrapolate to other lenders, products or economic conditions.
- **Right-censoring.** Some 60-month loans issued in 2010-2011 were still active at the May 2016 snapshot. Their charge-off rates are understated because not all outcomes are known. Compare vintages like-for-like (36-month loans) or by months on book.
- **No payment history.** There are no monthly balances or days-past-due fields. True delinquency buckets, cure rates and roll rates cannot be measured. The late-fee and prior-delinquency views here are proxies, and late fees can occur just before default.
- **Definition of default.** Charged off is a late, accounting-based event (typically well after 90 days past due) and is not a regulatory default definition.
- **Endogenous risk grades.** The grade and interest rate were set by the platform's own model and are not independent inputs. Differences by grade partly reflect that model and pricing.
- **Self-reported and mixed-quality inputs.** Income, purpose and employment details are largely self-reported; verification status varies.
- **Third-party copy.** The file is a public mirror of platform data, not an audited ledger. Reconcile to source before any formal use.
- **Stress tests are illustrative.** Scenario multipliers and the Monte Carlo correlation are assumptions, not calibrated forecasts. The stress book assumes today's observed segment PD/LGD apply to the whole origination mix.

## 7. Regulatory and decision-use limits
- Not for individual credit decisions: nothing here scores, approves or declines any applicant.
- Fair lending (e.g. ECOA / Regulation B, or local equivalents): state and income patterns can correlate with protected characteristics. Aggregate monitoring is fine; using these cuts to price or decline requires a separate fair-lending review.
- Adverse action and credit reporting rules (e.g. FCRA-type notice requirements) apply to real decisions and are not addressed here.
- Not an IFRS 9 / CECL / Basel provisioning or capital calculation. Segment PD and LGD here are simple observed averages without forward-looking adjustment or discounting.
- Model risk governance (e.g. SR 11-7-style validation) would be needed before any of this feeds a production model or limit.
- Use for portfolio monitoring, hypothesis generation and teaching. Re-run on the lender's own data before drawing operational conclusions.

## 8. Suggested monitoring actions
1. Track charge-off by vintage on a like-for-like basis (same term, same months on book) and report unresolved share next to every rate.
2. Set concentration triggers for single-state, single-purpose and single-vintage exposure (the HHI flags above are a starting point).
3. Collect monthly days-past-due and balance history so true delinquency buckets, cure rates and roll rates can replace today's proxies.
4. Re-run stress scenarios each period and compare the loss-rate swing by assumption; challenge the biggest ones first.


## Excel workbook (with a live stress calculator)

In [13]:
def write_workbook(path, scen, sens, reverse, mc, seg_out, vintage_term, vintage_grade_36, grade_mix, definitions):
    """Excel workbook: results tabs plus a live calculator with formulas."""
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        notes = pd.DataFrame(
            {"README": [
                "Scenario workbook for Project 12 (LendingClub 2007-2011 loans).",
                "Tabs: Scenarios, Sensitivity, ReverseStress, MonteCarlo, SegmentInputs, Calculator.",
                "Calculator tab has live formulas: change PD multiplier (B3) and LGD add-on (B4).",
                "Illustrative only: multipliers and correlations are assumptions, not forecasts.",
                "Formulas calculate when the file is opened in Excel or LibreOffice.",
            ]}
        )
        notes.to_excel(xw, sheet_name="README", index=False)
        scen.to_excel(xw, sheet_name="Scenarios", index=False)
        sens.to_excel(xw, sheet_name="Sensitivity", index=False)
        reverse.to_excel(xw, sheet_name="ReverseStress", index=False)
        mc.to_excel(xw, sheet_name="MonteCarlo", index=False)
        seg_out.to_excel(xw, sheet_name="SegmentInputs", index=False)
        vintage_term.to_excel(xw, sheet_name="Vintage_Term", index=False)
        vintage_grade_36.to_excel(xw, sheet_name="Vintage_Grade_36m", index=False)
        grade_mix.to_excel(xw, sheet_name="GradeMix_by_Vintage", index=False)
        definitions.to_excel(xw, sheet_name="DelinquencyDefinitions", index=False)

        ws = xw.book.create_sheet("Calculator")
        ws["A1"] = "Stress calculator: expected loss by grade x term"
        ws["A3"], ws["B3"] = "PD multiplier", 1.0
        ws["A4"], ws["B4"] = "LGD add-on (e.g. 0.10 = +10 pts)", 0.0
        headers = ["Grade", "Term (months)", "Exposure (USD)", "Base PD", "Base net LGD",
                   "Stressed PD", "Stressed LGD", "Expected loss (USD)", "EL % of exposure"]
        for j, h in enumerate(headers, 1):
            ws.cell(row=6, column=j, value=h)
        r = 7
        for row in seg_out.itertuples():
            ws.cell(row=r, column=1, value=row.grade)
            ws.cell(row=r, column=2, value=int(row.term_months))
            ws.cell(row=r, column=3, value=float(row.ead))
            ws.cell(row=r, column=4, value=float(row.pd) if pd.notna(row.pd) else 0.0)
            ws.cell(row=r, column=5, value=float(row.lgd_net) if pd.notna(row.lgd_net) else 0.0)
            ws.cell(row=r, column=6, value=f"=MIN(1,D{r}*$B$3)")
            ws.cell(row=r, column=7, value=f"=MIN(1,MAX(0,E{r}+$B$4))")
            ws.cell(row=r, column=8, value=f"=C{r}*F{r}*G{r}")
            ws.cell(row=r, column=9, value=f"=IF(C{r}=0,0,H{r}/C{r})")
            r += 1
        ws.cell(row=r, column=1, value="Total")
        ws.cell(row=r, column=3, value=f"=SUM(C7:C{r-1})")
        ws.cell(row=r, column=8, value=f"=SUM(H7:H{r-1})")
        ws.cell(row=r, column=9, value=f"=H{r}/C{r}")
        for sheet in xw.book.worksheets:
            for col in sheet.columns:
                width = max(len(str(c.value)) if c.value is not None else 0 for c in col)
                sheet.column_dimensions[col[0].column_letter].width = min(max(10, width + 2), 60)


write_workbook(os.path.join(OUTPUT_DIR, "scenario_workbook.xlsx"), scen, sens, reverse, mc, seg_out,
               vintage_term, vintage_grade_36, grade_mix, definitions)
print("Workbook written.")

Workbook written.


## Save all outputs and download

In [14]:
out = lambda n: os.path.join(OUTPUT_DIR, n)

audit.to_csv(out("1_validation_audit.csv"), index=False)
status_counts.to_csv(out("1_status_counts.csv"), index=False)
status_recon.to_csv(out("status_reconciliation.csv"), index=False)
rollfwd.to_csv(out("2_principal_rollforward.csv"), index=False)
definitions.to_csv(out("2_delinquency_definitions.csv"), index=False)
overlap.to_csv(out("2_definition_overlap.csv"), index=False)
features.to_csv(out("3_feature_dictionary.csv"), index=False)
stats.to_csv(out("4_portfolio_loss_stats.csv"), index=False)
quantiles.to_csv(out("4_quantiles.csv"), index=False)
perf.to_csv(out("5_performance_by_segment.csv"), index=False)
vintage_term.to_csv(out("5_vintage_by_term.csv"), index=False)
vintage_grade_36.to_csv(out("5_vintage_by_grade_36m.csv"), index=False)
grade_mix.to_csv(out("5_grade_mix_by_vintage.csv"), index=False)
conc_summary.to_csv(out("6_concentration_summary.csv"), index=False)
conc_groups.to_csv(out("6_concentration_groups.csv"), index=False)
single.to_csv(out("6_single_name_concentration.csv"), index=False)
loss_conc.to_csv(out("6_loss_concentration.csv"), index=False)
roll_proxy.to_csv(out("6_roll_proxy.csv"), index=False)
timing.to_csv(out("6_chargeoff_timing.csv"), index=False)
scen.to_csv(out("7_scenarios.csv"), index=False)
sens.to_csv(out("7_sensitivity.csv"), index=False)
reverse.to_csv(out("7_reverse_stress.csv"), index=False)
mc.to_csv(out("7_monte_carlo.csv"), index=False)
seg_out.to_csv(out("7_segment_inputs.csv"), index=False)
limits.to_csv(out("8_limitations.csv"), index=False)
with open(out("risk_memo.md"), "w", encoding="utf-8") as fh:
    fh.write(memo)

print("Files saved:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", f)

Files saved:
 - 1_status_counts.csv
 - 1_validation_audit.csv
 - 2_definition_overlap.csv
 - 2_delinquency_definitions.csv
 - 2_principal_rollforward.csv
 - 3_feature_dictionary.csv
 - 4_portfolio_loss_stats.csv
 - 4_quantiles.csv
 - 5_grade_mix_by_vintage.csv
 - 5_performance_by_segment.csv
 - 5_vintage_by_grade_36m.csv
 - 5_vintage_by_term.csv
 - 6_chargeoff_timing.csv
 - 6_concentration_groups.csv
 - 6_concentration_summary.csv
 - 6_loss_concentration.csv
 - 6_roll_proxy.csv
 - 6_single_name_concentration.csv
 - 7_monte_carlo.csv
 - 7_reverse_stress.csv
 - 7_scenarios.csv
 - 7_segment_inputs.csv
 - 7_sensitivity.csv
 - 8_limitations.csv
 - risk_memo.md
 - scenario_workbook.xlsx
 - status_reconciliation.csv


In [15]:
import shutil
zip_path = shutil.make_archive(os.path.join(BASE, "project12_outputs"), "zip", OUTPUT_DIR)
print("Zipped:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab; grab the zip from the path above.")

Zipped: /content/project12_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>